In [0]:
from pyspark.sql.functions import col, to_date, year, month, when, lower, trim, upper, datediff, when, lit, greatest
from pyspark.sql import functions as F

In [0]:
%sql
create schema if not exists 02_silver_catalog.silver 

In [0]:
bronze_schema = "01_bronze_catalog.bronze_azure_blob."
silver_schema = "02_silver_catalog.silver."

In [0]:
def normalize_columns(df):
    """Standardizes column names."""
    for c in df.columns:
        df = df.withColumnRenamed(c, c.strip().lower().replace(" ", "_"))
    return df

### Store table

In [0]:
df_store = spark.table(bronze_schema+"store")
df_store = normalize_columns(df_store)
df_store = (
    df_store

    .drop("_file", "_line", "_modified", "_fivetran_synced")
    .withColumn("store_id", F.col("store_id").cast("int"))
    .withColumn("opened_year", F.col("opened_year").cast("int"))
    .dropDuplicates()
)   
df_store.display()


In [0]:

manager_lookup = (
    df_store.filter(F.col("manager_name").isNotNull())
    .select("manager_id", F.col("manager_name").alias("manager_name_lookup")).distinct()
)
df_store = (
    df_store
    .join(manager_lookup, on="manager_id", how="left")
    .withColumn("manager_name", F.coalesce(F.col("manager_name"), F.col("manager_name_lookup")))
    .drop("manager_name_lookup")
)

store = df_store.withColumns({

    # Clean text
    "store_name": trim(col("store_name")),
    "city": lower(trim(col("city"))),
    "state": lower(trim(col("state"))),
    "manager_name": trim(col("manager_name")),
    "store_type": lower(trim(col("store_type"))),

    # Types
    "opened_year": col("opened_year").cast("int")
})

store.display()

In [0]:
store.write.format("delta").mode("overwrite").saveAsTable(silver_schema+"store")

### Order Table

In [0]:
df_order = spark.table(bronze_schema+"order")
df_order = normalize_columns(df_order)

df_order = (
    df_order
    .drop("_file", "_line", "_modified", "_fivetran_synced")
    .withColumn("store_id", F.col("store_id").cast("int"))
    .dropDuplicates()
)
df_order.display()


In [0]:
order = df_order.withColumns({
    #surrogate key

    # Clean text
    "service_type": lower(trim(col("service_type"))),
    "vehicle_make": lower(trim(col("vehicle_make"))),
    "vehicle_model": lower(trim(col("vehicle_model"))),
    "customer_name": trim(col("customer_name")),
    "technician_name": trim(col("technician_name")),
    "order_status": lower(trim(col("order_status"))),

    # Derived metrics (NULL-safe + controlled, handle negative dateflows)
    # greatest(x, y) returns the largest value between x and y
    "days_in_shop": when(
        col("actual_delivery_datetime").isNotNull() & col("vehicle_in_datetime").isNotNull(),
        greatest(datediff(col("actual_delivery_datetime"), col("vehicle_in_datetime")), lit(0))
    ).otherwise(lit(0)),

    "work_completion_to_delivery_days": when(
        col("actual_delivery_datetime").isNotNull() & col("actual_completion_datetime").isNotNull(),
        greatest(datediff(col("actual_delivery_datetime"), col("actual_completion_datetime")), lit(0))
    ).otherwise(lit(0)),

    "work_start_to_completion_days": when(
        col("actual_completion_datetime").isNotNull() & col("actual_work_start_datetime").isNotNull(),
        greatest(datediff(col("actual_completion_datetime"), col("actual_work_start_datetime")), lit(0))
    ).otherwise(lit(0)),

    "vehicle_in_to_work_start_days": when(
        col("actual_work_start_datetime").isNotNull() & col("vehicle_in_datetime").isNotNull(),
        greatest(datediff(col("actual_work_start_datetime"), col("vehicle_in_datetime")), lit(0))
    ).otherwise(lit(0)),

    "planned_vs_actual_completion_days": when(
        col("planned_completion_datetime").isNotNull() & col("actual_completion_datetime").isNotNull(),
        greatest(datediff(col("actual_completion_datetime"), col("planned_completion_datetime")), lit(0))
    ).otherwise(lit(0)),

    "promised_vs_actual_delivery_days": when(
        col("promised_delivery_datetime").isNotNull() & col("actual_delivery_datetime").isNotNull(),
        greatest(datediff(col("actual_delivery_datetime"), col("promised_delivery_datetime")), lit(0))
    ).otherwise(lit(0))
    
})



In [0]:
order.write.format("delta").option("mergeSchema","true").mode("overwrite").saveAsTable(silver_schema+"order")

### Customer Survey Table

In [0]:
df_customer_survey = spark.table(bronze_schema+"customer_survey")
df_customer_survey = normalize_columns(df_customer_survey)

customer_survey = (
    df_customer_survey
    .drop("_file", "_line", "_modified", "_fivetran_synced")
    .withColumn("delivered_on_time_rating", F.col("delivered_on_time_rating").cast("int"))
    .withColumn("work_quality_rating", F.col("work_quality_rating").cast("int"))
    .withColumn("cleanliness_rating", F.col("cleanliness_rating").cast("int"))
    .withColumn("communication_rating", F.col("communication_rating").cast("int"))
    .withColumn("overall_satisfaction_rating", F.col("overall_satisfaction_rating").cast("int"))
    .withColumn("survey_sent_ts", col("survey_sent_date"))
    .withColumn("survey_response_ts", col("survey_response_date"))
    .withColumn("survey_sent_date", to_date(col("survey_sent_date")))
    .withColumn("survey_response_date", to_date(col("survey_response_date")))
    .withColumn("responded_flag",F.when(F.col("responded_flag").isNull() & F.col("survey_response_date").isNotNull(), F.lit(True)).when(F.col("responded_flag").isNull() & F.col("survey_response_date").isNull(), F.lit(False))
    .otherwise(F.col("responded_flag"))
)
    .dropDuplicates()
)

display(customer_survey)

In [0]:

rating_cols = [
    "delivered_on_time_rating", "work_quality_rating",
    "cleanliness_rating", "communication_rating", "overall_satisfaction_rating"
]

# Join with order table to get store_id
df_order = spark.table(silver_schema+"order").select("order_id", "store_id")
customer_survey = customer_survey.join(df_order, on="order_id", how="left")

# Compute store-level average ratings (from responded surveys only)
store_avg = (
    customer_survey.filter(F.col("responded_flag") == True)
    .groupBy("store_id")
    .agg(*[F.round(F.avg(c)).cast("int").alias(f"{c}_avg") for c in rating_cols])
)

# Compute global averages as fallback
global_avg = (
    customer_survey.filter(F.col("responded_flag") == True)
    .agg(*[F.round(F.avg(c)).cast("int").alias(f"{c}_global") for c in rating_cols])
)

# Join store averages and fill nulls
customer_survey = customer_survey.join(store_avg, on="store_id", how="left")
customer_survey = customer_survey.crossJoin(global_avg)

for c in rating_cols:
    customer_survey = customer_survey.withColumn(
        c,
        F.coalesce(F.col(c), F.col(f"{c}_avg"), F.col(f"{c}_global"))
    )

# Drop helper columns and store_id (not in original schema)
cols_to_drop = [f"{c}_avg" for c in rating_cols] + [f"{c}_global" for c in rating_cols] + ["store_id"]
customer_survey = customer_survey.drop(*cols_to_drop)

In [0]:
customer_survey.write.format("delta").mode("overwrite").saveAsTable(silver_schema+"customer_survey")

### Estimate Table


In [0]:
df_estimate = spark.table(bronze_schema+"estimate")
df_estimate = normalize_columns(df_estimate)

df_estimate = (
    df_estimate
    .drop("_file", "_line", "_modified", "_fivetran_synced")
    .withColumn("version_no", F.col("version_no").cast("int"))
    .dropDuplicates()
)
df_estimate.display()

In [0]:

estimate = df_estimate.withColumns({

    # Preserve timestamp
    "created_ts": col("created_at"),

    # Add date
    "created_date": to_date(col("created_at")),

    # Clean text
    "estimator_name": trim(col("estimator_name")),
    "estimate_type": lower(trim(col("estimate_type"))),

    # Amount
    "estimate_amount": col("estimate_amount").cast("double")
})

display(estimate)

In [0]:
estimate.write.format("delta").option("mergeSchema","true").mode("overwrite").saveAsTable(silver_schema+"estimate")

### Invoice Table

In [0]:
df_invoice = spark.table(bronze_schema+"invoice")
df_invoice = normalize_columns(df_invoice)

invoice = (
    df_invoice
    .drop("_file", "_line", "_modified", "_fivetran_synced")
    .withColumn("invoice_amount", F.col("invoice_amount").cast("double"))
    .dropDuplicates()
)
invoice.display()

In [0]:
invoice = invoice.withColumns({

    # Preserve timestamp
    "invoice_ts": col("invoice_date"),

    # Add date
    "invoice_date": to_date(col("invoice_date")),

    # Clean text
    "payment_mode": lower(trim(col("payment_mode"))),
    "currency": upper(trim(col("currency"))),

    # Amount
    "invoice_amount": col("invoice_amount").cast("double")
})

display(invoice)

In [0]:
invoice.write.format("delta").mode("overwrite").saveAsTable(silver_schema+"invoice")

### NS Budget Table

In [0]:
df_ns_budget = spark.table(bronze_schema+"ns_budget")
ns_budget = normalize_columns(df_ns_budget)

ns_budget = (
    ns_budget
    .drop("_file", "_line", "_modified", "_fivetran_synced")
    .withColumn("ns_store_id", F.col("ns_store_id").cast("int"))
    .withColumn("budget_amount", F.col("budget_amount").cast("double"))
    .dropDuplicates()
)
ns_budget.display()

In [0]:

ns_budget = ns_budget.withColumns({

    # Rename + convert
    "budget_date": to_date(col("month")),

    # Derived columns
    "budget_year": year(to_date(col("month"))),
    "budget_month": month(to_date(col("month"))),

    # Amount
    "budget_amount": col("budget_amount").cast("double")
}).drop("month")


display(ns_budget)

In [0]:
ns_budget.write.format("delta").mode("overwrite").saveAsTable(silver_schema+"ns_budget")